In [ ]:
%run ../Module4/module4.ipynb


ImportError: attempted relative import with no known parent package

In [ ]:
def initialize_velocities(N, T_target):
    """
    Assigns initial velocities from a Maxwell-Boltzmann distribution.
    Generates a (N, 3) array with std_dev = sqrt(T_target),
    then removes net momentum so the center-of-mass is stationary.
    """
    sigma = np.sqrt(T_target)
    vel = np.random.normal(loc=0.0, scale=sigma, size=(N, 3))
    vel -= vel.mean(axis=0)  # zero net momentum
    return vel


In [ ]:
def rescale_velocities ( vel , T_target ) :

    """
    Scales velocities to match the target temperature .
    """
    # 1. Calculate current T_inst = (sum v^2) / 3N
    # 2. Scale factor lambda = sqrt ( T_target / T_inst )
    # 3. vel = vel * lambda
    T_inst = np.sum(vel**2) / (3 * len(vel))
    lambda_factor = np.sqrt(T_target / T_inst)
    vel *= lambda_factor
    return vel


In [ ]:
# --- THE MAIN MD LOOP ---
# 1. Setup: Define n_cells, density, T_target, dt, and steps.
# 2. Initialize: Call generate_fcc_lattice and initialize_velocities.
# 3. Main Loop:
#    - Call velocity_verlet_step (from Module 4)
#    - If in equilibration and step % 10 == 0: call rescale_velocities
#    - Accumulate energy and temperature data for plotting.

n_cells     = 4        # N = 4 * 4^3 = 256 atoms
rho_star    = 0.8442
T_target    = 1.0
dt          = 0.005
total_steps = 1000
r_cutoff    = 2.5;  r_cutoff_sq = r_cutoff**2

# Initialize positions and velocities
pos, L = generate_fcc_lattice(n_cells, rho_star)
N      = len(pos)
vel    = initialize_velocities(N, T_target)

# Initial forces
pe, force = calculate_total_force(pos, L, r_cutoff_sq)

energies     = []
temperatures = []

for step in range(total_steps):
    pos, vel, pe, force = velocity_verlet_step(pos, vel, force, dt, L, r_cutoff_sq)

    # Rescale velocities every 10 steps during equilibration (first half)
    if step < total_steps // 2 and step % 10 == 0:
        vel = rescale_velocities(vel, T_target)

    ke = calculate_kinetic_energy(vel)
    energies.append(ke + pe)
    temperatures.append(2 * ke / (3 * N))

# Plot energy and temperature
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(energies)
plt.title('Total Energy vs Time Steps')
plt.xlabel('Time Steps')
plt.ylabel('Total Energy')
plt.subplot(1, 2, 2)
plt.plot(temperatures)
plt.title('Temperature vs Time Steps')
plt.xlabel('Time Steps')
plt.ylabel('Temperature')
plt.tight_layout()
plt.show()
